# CIFAR-10 Classification

## 1.3 ResNet 18 Architecture + Data Augmentation

In [78]:
import torch
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import torch.optim as optim
import matplotlib.pyplot as plt
import time
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.nn.functional as F
import shutil

In [79]:
import numpy as np
import random
import os

seed = 42
# Setting Python and NumPy seeds
random.seed(seed)
np.random.seed(seed)
os.environ['PYTHONHASHSEED'] = str(seed)
    
# Setting PyTorch seeds (CPU and all GPUs)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
    
# Ensure deterministic behavior in cuDNN
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [80]:
TRAIN_MODEL = True
SAVE_MODEL = True

In [81]:
# checking if CUDA is available and set the device accordingly,
# if a GPU is available it will use it for computations, otherwise it will fall back to using the CPU.
# in this case I am using RTX 3060 GPU which has CUDA support
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

# Assuming that we are on a CUDA machine, this should print a CUDA device:
print(device)

cuda:0


## PreProcessing


In [82]:
train_transform = transforms.Compose(
    [
        # adding random cropping by increasing the padding to 4 so the size
        # of the image increases from 32x32 to 40x40 then randomly cropping
        # the image to 32x32 again
        transforms.RandomCrop(32, padding=4),
        # horizontal flipping each image by 50% probability meaning each image has
        # 50% probability of being flipped horizontaly
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ToTensor(),
        # using the dataset mean and std instead of using 0.5 to keep it in range of 0-1
        transforms.Normalize((0.4914, 0.4822, 0.4465),
                (0.2023, 0.1994, 0.2010))
    ]
)

test_transform = transforms.Compose(
    [
        # not applying augmentation to test dataset because we want the test
        # data not to be noisy and unfair we only want to introduce this to
        # the training dataset
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465),
            (0.2023, 0.1994, 0.2010))
    ]
)

batch_size = 64

trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                        download=False, transform=train_transform)
trainloader = DataLoader(trainset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)

testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                        download=False, transform=test_transform)
testloader = DataLoader(testset, batch_size=batch_size, shuffle=False, num_workers=2)

classes = ('plane', 'car', 'bird', 'cat',
           'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

## Resnet 18 Architecture

![Cifar Res Net 18](resnet18_full_visual_flow_readable.png)

In [83]:
# Reusable Convolution layer with 3x3 kernel and same padding, I am going to use batch normalization
# after each time I use this layer so bias = false is important 
def conv3x3(in_channels, out_channels, stride=1):
    return nn.Conv2d(
        in_channels=in_channels,
        out_channels=out_channels,
        kernel_size=3,
        stride=stride,
        padding=1,
        bias=False
    )

# Reusable Convolution layer with 1x1 kernel and same padding, I am going to use batch normalization
# after each time I use this layer so bias = false is important
def conv1x1(in_channels, out_channels, stride=1):
    return nn.Conv2d(
        in_channels=in_channels,
        out_channels=out_channels,
        kernel_size=1,
        stride=stride,
        bias=False
    )

In [84]:
# Constructing Basic Block this Network will be mainly for educational purposes 
# not generalization as it will only focus on ResNet 18 so Bottleneck Block
# is unnecessary

class BasicBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, downsample=None):
        super().__init__()
        
        # ResNet 18 applies 2 convolution layers before adding the skip or shortcut connection to the
        # new function
        # this block has these 2 convolution layers with batch normalization and Relu layers
        self.conv1 = conv3x3(in_channels=in_channels, out_channels=out_channels, stride=stride)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        
        self.conv2 = conv3x3(in_channels=out_channels, out_channels=out_channels)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        self.downsample = downsample
    
    def forward(self, x):
        # our identity or orginal image before the 2 convoloution layers
        identity = x 
        
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        
        out = self.conv2(out)
        out = self.bn2(out)
        
        # if there's a downsample block we apply it to the identity to increase the number of channels
        # or decrease the spatial size
        if self.downsample is not None:
            identity = self.downsample(x)
        
        # Image After 2 Convolution Layers + The Original Image Prior to the layers
        out = out + identity
        # we apply relu here to introduce non-linearity to the final output
        out = self.relu(out)
        
        return out

In [85]:
class ResNet(nn.Module):
    def __init__(self, num_classes=10): # num classes is set to 10 because the Cifar Dataset has only 10 labels for prediciton
        super().__init__()
        
        # Here out archeticture suggests that the first layer should have an input of 64 features
        self.in_channels = 64
        
        # this is the stem convolution layer which we learn basic information about our
        # original image then we output the same spatial dimenstion but 64 channels instead of 3
        # for our Basic Blocks
        self.conv1 = nn.Conv2d(
            in_channels=3,
            out_channels=64,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False
            )
        
        self.bn1 = nn.BatchNorm2d(64)
        
        # in_place just makes pytorch overwrite the original tensor instead of creating a new one
        # it saves a bit of memory that's why it's used here
        self.relu = nn.ReLU(inplace=True)
        
        # our 4 Stages or Layers that are the Base Blocks for ResNet 18
        # Here we make the image from the stem layer go through 4 layers with
        # various amount of blocks in this case it's 2 for each to train different
        # parameters
        # 64 x 32 x 32 -> 64 x 32 x 32 -> 128 x 16 x 16 -> 256 x 8 x 8 -> 512 x 4 x 4
        self.layer1 = self.make_layer(out_channels=64, num_blocks=2, stride=1)
        self.layer2 = self.make_layer(out_channels=128, num_blocks=2, stride=2)
        self.layer3 = self.make_layer(out_channels=256, num_blocks=2, stride=2)
        self.layer4 = self.make_layer(out_channels=512, num_blocks=2, stride=2)
        
        # at the end of our cnn layers we make the image go through avg pooling layer to change
        # it's spatial dimenstions from 512 x 4 x 4 to 512 x 1 x 1
        self.avgpool = nn.AdaptiveAvgPool2d((1,1))
        # final fully connected layers to predict logits for each class
        self.fc1 = nn.Linear(512, num_classes)
    
    # function to create a new stage or layer for ResNet
    def make_layer(self, out_channels, num_blocks, stride):
        # not downsampling in default 
        downsample = None
        
        # if stride is not 1 meaning the spatial dimentions of the has changed or
        # if number of input channels is not equal to out channels meaning
        # number of channels aren't the same for example the skip connection from layer 1 had an output
        # of 64 channels and 32 x 32 spatial dimensions but we want to add it to an image with
        # 128 channels and 16 x 16 spatial dimensions this is not possible so we have to downsample
        # the skip connection to make it fit
        if stride != 1 or self.in_channels != out_channels:
            downsample = nn.Sequential(
                # 1x1 convolutions are useful for changing channels and spatial dimensions because
                # it doesn't focus on neighboring pixels just how to change each pixel based on the
                # current channels
                conv1x1(self.in_channels, out_channels, stride),
                # we pass it through batch norm layer to normalize the outputs
                nn.BatchNorm2d(out_channels)
            )
        
        # Creating a layers list to store all blocks
        layers = []
        # this is the first block our main focus here is to change the input's channels and spatial
        # dimenstion and learn some features 
        layers.append(BasicBlock(
            in_channels=self.in_channels,
            out_channels=out_channels,
            stride=stride,
            downsample=downsample
            ))
        
        # since the number of input channels are equal to the number of output channels by default
        # or by downsampling we set in_channels equal to out_channels to pass it along the layers
        self.in_channels = out_channels
        
        # for loop to pass the output of the first block through the rest of the blocks
        # here the spatial dimenstions and channels are the same because we already changed
        # it in the first block 
        
        # in this instance this for loop will run only once because we have num_blocks set to 2
        # in each layer
        for _ in range(1, num_blocks):
            layers.append(BasicBlock(
                in_channels=self.in_channels,
                out_channels=out_channels
                ))
        
        # since nn.Sequential expects multiple arugments not a list we unpack layers using 
        # *layers to turn it into arguments [block1, block2] -> (block1, block2)
        return nn.Sequential(*layers)
    
    def forward(self, x):
        # just the basic flow of the image through the neural network from the ResNet 18 Graph
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc1(x)
        
        return x

Cifar_10_ResNet = ResNet(num_classes=10)

Cifar_10_ResNet.to(device)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=Fals

In [86]:
# Using Adam Optimizer with 0.001 Learning Rate and 0.0001 l2 alpha/weight decay
optimizer = optim.Adam(Cifar_10_ResNet.parameters(), lr=1e-3, weight_decay=1e-4)

criterion = nn.CrossEntropyLoss()

In [87]:
if TRAIN_MODEL:
    # Train model
    train_loss=[]
    train_accuary=[]
    test_loss=[]
    test_accuary=[]

    num_epochs = 20   #(set no of epochs)
    start_time = time.time() #(for showing time)
    # Start loop
    for epoch in range(num_epochs): #(loop for every epoch)
        print("Epoch {} running".format(epoch)) #(printing message)
        """ Training Phase """
        Cifar_10_ResNet.train()    #(training model)
        running_loss = 0.   #(set loss 0)
        running_corrects = 0 
        # load a batch data of images
        for i, (inputs, labels) in enumerate(trainloader):
            inputs = inputs.to(device)
            labels = labels.to(device) 
            # forward inputs and get output
            optimizer.zero_grad()
            outputs = Cifar_10_ResNet(inputs)
            _, preds = torch.max(outputs, 1)
            loss = criterion(outputs, labels)
            # get loss value and update the network weights
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            running_corrects += torch.sum(preds == labels.data).item()
        epoch_loss = running_loss / len(trainset)
        epoch_acc = running_corrects / len(trainset) * 100.
        # Append result
        train_loss.append(epoch_loss)
        train_accuary.append(epoch_acc)
        # Print progress
        print('[Train #{}] Loss: {:.4f} Acc: {:.4f}% Time: {:.4f}s'.format(epoch+1, epoch_loss, epoch_acc, time.time() -start_time))
        """ Testing Phase """
        Cifar_10_ResNet.eval()
        with torch.no_grad():
            running_loss = 0.
            running_corrects = 0
            for inputs, labels in testloader:
                inputs = inputs.to(device)
                labels = labels.to(device)
                outputs = Cifar_10_ResNet(inputs)
                _, preds = torch.max(outputs, 1)
                loss = criterion(outputs, labels)
                running_loss += loss.item()
                running_corrects += torch.sum(preds == labels.data).item()
            epoch_loss = running_loss / len(testset)
            epoch_acc = running_corrects / len(testset) * 100.
            # Append result
            test_loss.append(epoch_loss)
            test_accuary.append(epoch_acc)
            # Print progress
            print('[Test #{}] Loss: {:.4f} Acc: {:.4f}% Time: {:.4f}s'.format(epoch+1, epoch_loss, epoch_acc, time.time()- start_time))

Epoch 0 running
[Train #1] Loss: 0.0226 Acc: 46.9500% Time: 39.1931s
[Test #1] Loss: 0.0193 Acc: 57.4900% Time: 47.6028s
Epoch 1 running
[Train #2] Loss: 0.0157 Acc: 64.2600% Time: 87.7769s
[Test #2] Loss: 0.0186 Acc: 62.3100% Time: 96.4386s
Epoch 2 running
[Train #3] Loss: 0.0128 Acc: 70.9100% Time: 137.6482s
[Test #3] Loss: 0.0146 Acc: 70.2800% Time: 146.4534s
Epoch 3 running
[Train #4] Loss: 0.0108 Acc: 76.0060% Time: 188.6996s
[Test #4] Loss: 0.0145 Acc: 69.2700% Time: 197.6519s
Epoch 4 running
[Train #5] Loss: 0.0095 Acc: 79.1040% Time: 240.4188s
[Test #5] Loss: 0.0100 Acc: 78.8700% Time: 249.4509s
Epoch 5 running
[Train #6] Loss: 0.0085 Acc: 81.3160% Time: 292.8649s
[Test #6] Loss: 0.0090 Acc: 80.6900% Time: 302.0748s
Epoch 6 running
[Train #7] Loss: 0.0078 Acc: 82.9800% Time: 346.4814s
[Test #7] Loss: 0.0083 Acc: 82.2300% Time: 357.5484s
Epoch 7 running
[Train #8] Loss: 0.0071 Acc: 84.4480% Time: 403.6532s
[Test #8] Loss: 0.0085 Acc: 82.1900% Time: 412.8716s
Epoch 8 running
[Tra

In [ ]:
if SAVE_MODEL:
    PATH = "./models/cifar_resnet_scratch.pth"
    torch.save(Cifar_10_ResNet, PATH)


In [93]:
if SAVE_MODEL:
    train_loss_table = pd.DataFrame(train_loss, columns=["Loss"])
    train_loss_table["Epoch"] = train_loss_table.index + 1

    train_accuracy_table = pd.DataFrame(train_accuary, columns=["Accuracy"])
    train_accuracy_table["Epoch"] = train_accuracy_table.index + 1
    
    test_loss_table = pd.DataFrame(test_loss, columns=["Loss"])
    test_loss_table["Epoch"] = test_loss_table.index + 1
    
    test_accuracy_table = pd.DataFrame(test_accuary, columns=["Accuracy"])
    test_accuracy_table["Epoch"] = test_accuracy_table.index + 1
    
    train_loss_table.to_csv("tables/ResNet_Scratch/training_loss.csv", index=False)
    train_accuracy_table.to_csv("tables/ResNet_Scratch/training_accuracy.csv", index=False)
    
    test_loss_table.to_csv("tables/ResNet_Scratch/test_loss.csv", index=False)
    test_accuracy_table.to_csv("tables/ResNet_Scratch/test_accuracy.csv", index=False)